# DX 704 Week 12 Project

This week's project will revisit the email spam classifier project from week 9 using large language model embeddings instead of custom features.


The full project description and a template notebook are available on GitHub: [Project 12 Materials](https://github.com/bu-cds-dx704/dx704-project-12).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip"
data_zip_path = Path("enron_spam_data.zip")
if not data_zip_path.exists():
    urlretrieve(DATA_URL, data_zip_path)
data_zip_path

PosixPath('enron_spam_data.zip')

In [2]:
import pandas as pd

In [3]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [4]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Download BERT Model

We will use a pre-trained BERT model to extract embedding vectors as described in lesson 2.1 this week.
Here is sample code to download the model from [Hugging Face](https://huggingface.co/) and extract one vector.
This model is small enough that you can run it with CPU only, but GPUs will be faster if available.

In [5]:
# You may need to install torch and transformers.
# Google Colab will have these installed already.
#
# pip install transformers torch --upgrade

import torch
from transformers import AutoTokenizer, AutoModel

/Users/tetianakravchuk/Desktop/BUProjects/DX704-AI-in-the-Field/project-12-main/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
device

'mps'

In [7]:
MODEL_NAME = "bert-base-uncased"
MODEL_CACHE_DIR = ".hf-cache"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=MODEL_CACHE_DIR)
bert_model = AutoModel.from_pretrained(MODEL_NAME, cache_dir=MODEL_CACHE_DIR)
bert_model.to(device)
bert_model.eval()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22613.56it/s]


BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [8]:
@torch.no_grad()
def embed_text(text):
    batch = [text]
    inputs = tokenizer(batch, padding=True, truncation=True, return_tensors="pt").to(device)
    outputs = bert_model(**inputs)
    # CLS token embedding is the first token's hidden state
    cls_emb = outputs.last_hidden_state[:, 0, :]  # shape: [batch_size, 768]
    return cls_emb.cpu()

In [9]:
v = embed_text("Hi, will you buy my spam?")
v.shape

torch.Size([1, 768])

## Part 3: Create Embedding Vectors

Use BERT to create embeddings for each email in the Enron data set.
You will have to decide how to combine the different columns of the original data set to produce one embedding vector.


Hint: BERT can be run without a GPU, but will be much slower.
Using Google Colab with only a CPU, it runs around 1 embedding per second.
Using Google Colab with the T4 GPU option, it runs around 60 embeddings per second.
Caching is also encouraged to avoid unnecessary reruns.

In [10]:
import json

import numpy as np

EMBEDDING_CACHE_PATH = Path("embedding_cache.npy")
EMBEDDING_BATCH_SIZE = 64

def build_email_text(row):
    subject = "" if pd.isna(row["Subject"]) else str(row["Subject"]).strip()
    message = "" if pd.isna(row["Message"]) else str(row["Message"]).strip()
    pieces = [piece for piece in [subject, message] if piece]
    return f" {tokenizer.sep_token} ".join(pieces)

@torch.no_grad()
def embed_texts(texts, batch_size=EMBEDDING_BATCH_SIZE):
    batches = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)
        outputs = bert_model(**inputs)
        batches.append(outputs.last_hidden_state[:, 0, :].cpu())
    return torch.cat(batches, dim=0)

email_texts = enron_spam_data.apply(build_email_text, axis=1).tolist()
expected_shape = (len(email_texts), bert_model.config.hidden_size)

if EMBEDDING_CACHE_PATH.exists():
    cached_embeddings = np.load(EMBEDDING_CACHE_PATH)
    if cached_embeddings.shape == expected_shape:
        embedding_vectors = cached_embeddings.astype("float32", copy=False)
    else:
        embedding_vectors = embed_texts(email_texts).numpy().astype("float32")
        np.save(EMBEDDING_CACHE_PATH, embedding_vectors)
else:
    embedding_vectors = embed_texts(email_texts).numpy().astype("float32")
    np.save(EMBEDDING_CACHE_PATH, embedding_vectors)

embedding_vectors.shape

(33716, 768)

Save your embeddings in a file "embeddings.tsv.gz" with two columns, Message ID and embedding_vector_json, where embedding_vector_json is a JSON-encoded list.
Make sure that embedding_vector_json is a 1 dimensional list, not 2 dimensional.

Hint: don't forget the ".gz" extension indicating gzip compression.
The Pandas `.to_csv` method will automatically add the compression if you save data with a filename ending in ".gz", so you just need to pass it the right filename.

Hint: Gradescope only allows files up to 100MB to be submitted.
Round your embeddings to 3 decimal places to make them smaller.

In [11]:
rounded_embeddings = np.round(embedding_vectors, 3)
embeddings_output = pd.DataFrame(
    {
        "Message ID": enron_spam_data["Message ID"].astype(int),
        "embedding_vector_json": [
            json.dumps(row.tolist(), separators=(",", ":"))
            for row in rounded_embeddings
        ],
    }
)
embeddings_output.to_csv("embeddings.tsv.gz", sep="\t", index=False)
embeddings_output.head()

,Message ID,embedding_vector_json
0,0,"[-0.4020000100135803,0.29499998688697815,-0.10..."
1,1,"[-0.5569999814033508,-0.03099999949336052,0.13..."
2,2,"[-0.5870000123977661,-0.12999999523162842,-0.0..."
3,3,"[-0.7429999709129333,-0.2549999952316284,-0.16..."
4,4,"[-0.7459999918937683,-0.21400000154972076,0.28..."


Submit "embeddings.tsv.gz" in Gradescope.

## Part 4: Train a Linear Regression

Train an ordinary least squares regression for spam/ham status where spam is treated as target value 1 and ham is treated as target value 0 with your embeddings above as the only input variables.


In [12]:
X = embedding_vectors.astype("float64")
y = (enron_spam_data["Spam/Ham"] == "spam").astype("float64").to_numpy()
X_mean = X.mean(axis=0)
y_mean = y.mean()
linear_coefficients, *_ = np.linalg.lstsq(X - X_mean, y - y_mean, rcond=None)
linear_intercept = float(y_mean - X_mean @ linear_coefficients)
linear_intercept, linear_coefficients.shape

(2682.5687954849677, (768,))

Save the coefficients of your linear model in a file "coefficients.tsv" with columns dim and coefficient where dim specifies the offset in the embedding vector (0-767).
Don't worry about the bias term (but your model should still have it).

In [13]:
coefficients_output = pd.DataFrame(
    {
        "dim": np.arange(linear_coefficients.size, dtype=int),
        "coefficient": linear_coefficients,
    }
)
coefficients_output.to_csv("coefficients.tsv", sep="\t", index=False)
coefficients_output.head()

,dim,coefficient
0,0,184.065047
1,1,186.481232
2,2,176.469618
3,3,191.739154
4,4,184.309128


Submit "coefficients.tsv" in Gradescope.

## Part 5: Search for Relevant Documents

The file "queries.tsv" specifies 10 queries.
For each of the queries, encode them as a vector, and find the message that is closest using $L_2$.

In [14]:
queries = pd.read_csv("queries.tsv", sep="\t")
query_vectors = embed_texts(queries["query"].tolist()).numpy().astype("float64")
document_sq_norms = (X**2).sum(axis=1, keepdims=True)
query_sq_norms = (query_vectors**2).sum(axis=1)
distance_sq = document_sq_norms + query_sq_norms - 2 * (X @ query_vectors.T)
best_match_index = np.argmin(np.maximum(distance_sq, 0.0), axis=0)
query_matches = queries.copy()
query_matches["Message ID"] = enron_spam_data.iloc[best_match_index]["Message ID"].to_numpy()
query_matches["query_vector_json"] = [
    json.dumps(np.round(vec, 3).tolist(), separators=(",", ":"))
    for vec in query_vectors
]
query_matches[["query_id", "Message ID"]]

,query_id,Message ID
0,1,13950
1,2,14258
2,3,18071
3,4,13222
4,5,3743
5,6,21810
6,7,29544
7,8,14635
8,9,14635
9,10,21181


Save your results in a file "query-matches.tsv" with columns query_id, query_vector_json, and Message ID.

In [15]:
query_matches_output = query_matches[["query_id", "query_vector_json", "Message ID"]]
query_matches_output.to_csv("query-matches.tsv", sep="\t", index=False)
query_matches_output

,query_id,query_vector_json,Message ID
0,1,"[-0.067,0.486,-0.697,0.012,0.353,0.001,-0.1,0....",13950
1,2,"[-0.865,-0.246,0.211,-0.16,-0.095,0.163,0.608,...",14258
2,3,"[-0.298,-0.203,-0.198,0.104,-0.608,-0.122,0.37...",18071
3,4,"[-0.215,0.199,-0.022,0.005,0.379,-0.211,0.116,...",13222
4,5,"[-0.196,-0.049,-0.286,0.084,0.672,-0.281,0.622...",3743
5,6,"[-0.542,-0.275,-0.15,-0.028,0.283,-0.127,0.041...",21810
6,7,"[-0.124,0.354,0.073,-0.094,-0.022,-0.217,0.099...",29544
7,8,"[-0.258,0.056,-0.195,0.072,-0.175,0.306,0.162,...",14635
8,9,"[-0.32,0.158,-0.184,-0.192,0.056,0.259,0.194,0...",14635
9,10,"[-0.298,-0.306,-0.281,-0.208,0.501,0.198,0.172...",21181


Submit "query-matches.tsv" in Gradescope.

## Part 6: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 7: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

None.

Additional libraries: `json`, `pathlib`, and `urllib.request` from the Python standard library for output formatting, path handling, and downloading the data file; `numpy` for array handling and ordinary least squares.

Generative AI tools: OpenAI Codex in this working session. Transcript link: unavailable from the local notebook environment.